# Running Totals and Moving Averages

## Overview
Running totals and moving averages are computed by combining an aggregate window function with a **frame clause** that controls exactly which rows are included relative to the current row.

**Frame clause syntax:**
```sql
SUM(amount) OVER (
    PARTITION BY account_id
    ORDER BY txn_date
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
)
```

**Frame modes:**

| Mode | Row identification | Behaviour with ties |
|---|---|---|
| `ROWS` | Physical rows — by position | Ties treated as separate rows |
| `RANGE` | Logical rows — by value equality | All rows with the same ORDER BY value included |
| `GROUPS` | Groups of equal ORDER BY values | PostgreSQL 11+ only |

**Common frame specifications:**

| Frame | Meaning |
|---|---|
| `ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW` | All rows from start to current (cumulative) |
| `ROWS BETWEEN 6 PRECEDING AND CURRENT ROW` | Current row + 6 preceding (7-row rolling window) |
| `ROWS BETWEEN 2 PRECEDING AND 2 FOLLOWING` | Centred window: 2 before + current + 2 after |
| `ROWS BETWEEN CURRENT ROW AND UNBOUNDED FOLLOWING` | Current to end of partition |
| `ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING` | Entire partition (same as no ORDER BY) |

**Default frame when ORDER BY is present:** `RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW` — this is almost always what you want for running totals, but use `ROWS` explicitly to avoid RANGE surprises with ties.

---

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
conn.executescript("""
CREATE TABLE transactions (txn_id INTEGER PRIMARY KEY, account_id INTEGER, txn_date TEXT, txn_type TEXT, amount REAL, category TEXT, flagged INTEGER);
CREATE TABLE trades (trade_id INTEGER PRIMARY KEY, account_id INTEGER, trade_date TEXT, ticker TEXT, direction TEXT, shares INTEGER, price REAL, total_value REAL);
INSERT INTO transactions VALUES
  (1001,101,'2023-01-05','Deposit',4200.00,'Payroll',0),(1002,101,'2023-01-12','Withdrawal',-850.00,'Rent',0),
  (1003,101,'2023-01-20','Withdrawal',-120.00,'Groceries',0),(1004,101,'2023-02-05','Deposit',4200.00,'Payroll',0),
  (1005,101,'2023-02-18','Withdrawal',-980.00,'Rent',0),(1006,101,'2023-03-05','Deposit',4200.00,'Payroll',0),
  (1007,103,'2023-01-08','Deposit',15000.00,'Payroll',0),(1008,103,'2023-01-25','Withdrawal',-3200.00,'Rent',0),
  (1009,103,'2023-02-08','Deposit',15000.00,'Payroll',0),(1010,103,'2023-02-22','Withdrawal',-2800.00,'Rent',0),
  (1011,103,'2023-03-08','Deposit',15000.00,'Payroll',0),(1012,105,'2023-01-06','Deposit',3100.00,'Payroll',0),
  (1013,105,'2023-01-19','Withdrawal',-700.00,'Rent',0),(1014,105,'2023-02-06','Deposit',3100.00,'Payroll',0),
  (1015,105,'2023-02-20','Withdrawal',-650.00,'Groceries',0),(1016,107,'2023-01-20','Deposit',3800.00,'Payroll',0),
  (1017,107,'2023-02-10','Fee',-25.00,'Fee',1),(1018,107,'2023-03-15','Withdrawal',-450.00,'Groceries',1),
  (1019,109,'2023-01-10','Deposit',2800.00,'Payroll',0),(1020,109,'2023-01-28','Withdrawal',-650.00,'Groceries',0),
  (1021,111,'2023-01-22','Deposit',4500.00,'Payroll',0),(1022,111,'2023-02-15','Withdrawal',-1100.00,'Utilities',0),
  (1023,111,'2023-03-01','Deposit',4500.00,'Payroll',0);
INSERT INTO trades VALUES
  (2001,104,'2023-01-10','AAPL','Buy',10,148.50,1485.00),(2002,104,'2023-01-25','MSFT','Buy',5,240.10,1200.50),
  (2003,104,'2023-02-14','AAPL','Buy',5,153.20,766.00),(2004,104,'2023-02-28','MSFT','Sell',3,252.80,758.40),
  (2005,104,'2023-03-15','NVDA','Buy',2,278.50,557.00),(2006,108,'2023-01-05','AAPL','Buy',20,130.50,2610.00),
  (2007,108,'2023-01-18','AMZN','Buy',8,95.20,761.60),(2008,108,'2023-02-08','AAPL','Sell',10,151.00,1510.00),
  (2009,108,'2023-02-22','AMZN','Buy',5,99.80,499.00),(2010,108,'2023-03-10','NVDA','Buy',4,265.30,1061.20),
  (2011,110,'2023-01-12','MSFT','Buy',8,238.00,1904.00),(2012,110,'2023-02-01','AAPL','Buy',15,145.00,2175.00),
  (2013,110,'2023-02-20','MSFT','Buy',3,248.50,745.50),(2014,110,'2023-03-05','AAPL','Sell',10,155.00,1550.00),
  (2015,110,'2023-03-20','NVDA','Buy',6,280.00,1680.00);
""")
conn.commit()
print('Schema ready.')


---
## Running (cumulative) total

In [ ]:
# Cumulative running total per account
print('=== Running total of amount per account ===')
q = """
SELECT  txn_id,
        account_id,
        txn_date,
        txn_type,
        amount,
        ROUND(SUM(amount) OVER (
            PARTITION BY account_id
            ORDER BY txn_date, txn_id
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ), 2)  AS running_balance,
        COUNT(*) OVER (
            PARTITION BY account_id
            ORDER BY txn_date, txn_id
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        )      AS txn_sequence
FROM    transactions
WHERE   account_id IN (101, 103)
ORDER BY account_id, txn_date, txn_id
"""
print(pd.read_sql(q, conn).to_string(index=False))


---
## ROWS vs RANGE — why ties matter

In [ ]:
# Show the difference between ROWS and RANGE when dates are identical
# Add two same-date transactions to illustrate
conn.execute("INSERT INTO transactions VALUES (1099,101,'2023-01-05','Fee',-15.00,'Fee',0)")
conn.commit()

print('=== ROWS vs RANGE with tied dates (2 txns on 2023-01-05 for account 101) ===')
q = """
SELECT  txn_id,
        txn_date,
        amount,
        ROUND(SUM(amount) OVER (
            PARTITION BY account_id ORDER BY txn_date
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ), 2)  AS running_rows,
        ROUND(SUM(amount) OVER (
            PARTITION BY account_id ORDER BY txn_date
            RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ), 2)  AS running_range
FROM    transactions
WHERE   account_id = 101
ORDER BY txn_date, txn_id
"""
print(pd.read_sql(q, conn).to_string(index=False))
print()
print('ROWS: each physical row processed in sequence — running total grows row-by-row.')
print('RANGE: all rows with the same date value included together — both Jan 5 rows')
print('       see the same running total (sum of all Jan 5 rows included at once).')
print('Use ROWS for running totals to avoid the RANGE tie ambiguity.')

# Remove the extra row for remaining demos
conn.execute("DELETE FROM transactions WHERE txn_id = 1099")
conn.commit()


---
## Moving (rolling) averages

In [ ]:
# 3-transaction rolling average of amount per account
print('=== 3-row rolling average of transaction amount ===')
q = """
SELECT  txn_id,
        account_id,
        txn_date,
        amount,
        ROUND(AVG(amount) OVER (
            PARTITION BY account_id
            ORDER BY txn_date, txn_id
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW   -- current + 2 prior = 3-row window
        ), 2)  AS rolling_3_avg,
        COUNT(*) OVER (
            PARTITION BY account_id
            ORDER BY txn_date, txn_id
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        )      AS rows_in_window   -- will be < 3 at partition start
FROM    transactions
WHERE   account_id IN (101, 103)
ORDER BY account_id, txn_date, txn_id
"""
print(pd.read_sql(q, conn).to_string(index=False))
print()
print('rows_in_window < 3 at the start of each partition — window shrinks at edges.')
print('This is correct: you have fewer than 3 prior rows, so the average uses what exists.')


---
## Centred windows and forward-looking frames

In [ ]:
# Centred 3-row window: 1 preceding, current, 1 following
print('=== Centred 3-row moving average (smoothing) ===')
q = """
SELECT  trade_id,
        account_id,
        trade_date,
        ticker,
        price,
        ROUND(AVG(price) OVER (
            PARTITION BY account_id, ticker
            ORDER BY trade_date, trade_id
            ROWS BETWEEN 1 PRECEDING AND 1 FOLLOWING
        ), 2)  AS smoothed_price,
        COUNT(*) OVER (
            PARTITION BY account_id, ticker
            ORDER BY trade_date, trade_id
            ROWS BETWEEN 1 PRECEDING AND 1 FOLLOWING
        )      AS window_size
FROM    trades
ORDER BY account_id, ticker, trade_date
"""
print(pd.read_sql(q, conn).to_string(index=False))

# Running total from current row to end of partition (reverse cumulative)
print()
print('=== Remaining transactions from current row to end (CURRENT ROW to UNBOUNDED FOLLOWING) ===')
q2 = """
SELECT  txn_id,
        account_id,
        txn_date,
        amount,
        ROUND(SUM(amount) OVER (
            PARTITION BY account_id
            ORDER BY txn_date, txn_id
            ROWS BETWEEN CURRENT ROW AND UNBOUNDED FOLLOWING
        ), 2)  AS remaining_sum
FROM    transactions
WHERE   account_id = 101
ORDER BY txn_date, txn_id
"""
print(pd.read_sql(q2, conn).to_string(index=False))

conn.close()


---
## Common Pitfalls

**1. Default frame with ORDER BY is RANGE, not ROWS**
When you write `SUM(x) OVER (ORDER BY date)` without an explicit frame clause, the default is `RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW`. For dates with ties, RANGE includes all rows with the same date value in the "current" frame — producing the same running total for all tied rows. Always write `ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW` explicitly for running totals.

**2. Window shrinks at partition boundaries**
`ROWS BETWEEN 6 PRECEDING AND CURRENT ROW` uses fewer than 7 rows at the start of each partition. `AVG` over a 7-row window will average only 1, 2, 3... rows at the partition start. The `COUNT(*)` trick shown above lets you see the actual window size — important for weighted calculations.

**3. RANGE mode requires sortable, comparable types**
`RANGE BETWEEN INTERVAL '7 days' PRECEDING AND CURRENT ROW` (PostgreSQL) works with date/timestamp columns. SQLite does not support RANGE with value-based offsets — use `ROWS` for all frame-based work in SQLite.

**4. Running total depends entirely on ORDER BY determinism**
A running total using `ORDER BY txn_date` is non-deterministic if multiple transactions share the same date. The balance at any mid-date row may vary between executions. Always include a unique column in the ORDER BY: `ORDER BY txn_date, txn_id`.

**5. Moving average with ROWS doesn't align with calendar windows**
`ROWS BETWEEN 6 PRECEDING AND CURRENT ROW` is a 7-*row* window, not a 7-*day* window. If data is sparse (some days missing), the 7-row window may span more than a week. For true calendar-period windows, use a date spine with a range join or PostgreSQL's RANGE frame with an interval offset.

**6. Cumulative AVG is not the same as a rolling AVG**
`AVG(amount) OVER (ORDER BY date ROWS UNBOUNDED PRECEDING)` is a cumulative average (all rows from start). `AVG(amount) OVER (ORDER BY date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW)` is a 7-row rolling average. They answer different questions — be explicit about which you need.


---
*sql_methods_library - Samantha McGarrigle*